# 2. Image modality: anxiety from facial expressions (Gabor features + CNN)

Face images from two public facial-expression datasets are grouped into *anxiety-related* and *other* expressions, passed through a bank of Gabor filters and classified with a small convolutional network.

**Datasets** (not included in this repository because of their licences, see `data/README.md`):

- **CK+48**: 48x48 grayscale crops from the Extended Cohn-Kanade dataset, 7 emotions
- **KDEF**: Karolinska Directed Emotional Faces, 7 emotions, resized to 48x48 grayscale here

**Emotion to label mapping used in the thesis**

| Label | Emotions |
| --- | --- |
| Anxiety (1) | anger, contempt, disgust, sadness, neutral |
| No anxiety (0) | fear, happiness, surprise |

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")  # hide TensorFlow start-up logs
import tensorflow as tf
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import Conv2D, Dense, Dropout, Flatten, Input, MaxPool2D
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.utils import to_categorical

from preprocessing import gabor_features, load_face, to_model_input

IMAGES = Path("../data/images")
MODELS = Path("../models")
OUTPUTS = Path("../outputs")
MODELS.mkdir(exist_ok=True)
OUTPUTS.mkdir(exist_ok=True)

tf.random.set_seed(11)

## Load the images

In [ ]:
# folder name -> emotion, for each dataset
DATASETS = {
    "ck+48": {"anger": "anger", "contempt": "contempt", "disgust": "disgust", "fear": "fear",
              "happy": "happy", "sadness": "sadness", "surprise": "surprise"},
    "kdef": {"angry": "anger", "disgust": "disgust", "fear": "fear", "happy": "happy",
             "neutral": "neutral", "sad": "sadness", "surprise": "surprise"},
}
ANXIETY_EMOTIONS = {"anger", "contempt", "disgust", "sadness", "neutral"}
CLASSES = ["No anxiety", "Anxiety"]

records = []
for dataset, folders in DATASETS.items():
    for folder, emotion in folders.items():
        for path in sorted((IMAGES / dataset / folder).iterdir()):
            records.append({"path": path, "dataset": dataset, "emotion": emotion})

faces = pd.DataFrame(records)
faces["anxiety"] = faces["emotion"].isin(ANXIETY_EMOTIONS).astype(int)
images = np.stack([load_face(p) for p in faces["path"]])

print(images.shape)
pd.crosstab(faces["emotion"], faces["dataset"], margins=True)

In [ ]:
faces['anxiety'].map(dict(enumerate(CLASSES))).value_counts()

## Gabor features

Each face is filtered with six Gabor kernels (0 to 150 degrees in 30 degree steps, 11x11, sigma 1.5, wavelength 3) and the responses are summed. Gabor filters respond to oriented edges and textures such as wrinkles around the eyes, brows and mouth, which carry most of the expression information.

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for i, ax in enumerate(axes.T):
    idx = faces.groupby("emotion").head(1).index[i]
    ax[0].imshow(images[idx], cmap="gray")
    ax[0].set_title(faces.loc[idx, "emotion"])
    ax[1].imshow(gabor_features(images[idx]), cmap="gray")
for ax in axes.ravel():
    ax.axis("off")
plt.suptitle("Original faces (top) and Gabor features (bottom)")
plt.tight_layout()
plt.show()

## Train / validation / test split

The test set has 116 images so it can be paired one-to-one with the 116 questionnaire test responses in the fusion step. A further 10% of the training images is held out for early stopping, so the test set is never seen during training.

In [ ]:
idx_train, idx_test = train_test_split(faces.index, test_size=116, random_state=11, stratify=faces["anxiety"])
idx_train, idx_val = train_test_split(idx_train, test_size=0.1, random_state=11, stratify=faces.loc[idx_train, "anxiety"])

X_train, X_val, X_test = (to_model_input(images[i]) for i in (idx_train, idx_val, idx_test))
y_train, y_val, y_test = (to_categorical(faces.loc[i, "anxiety"], 2) for i in (idx_train, idx_val, idx_test))
print("Train:", X_train.shape, " Validation:", X_val.shape, " Test:", X_test.shape)

## Model

In [ ]:
def create_model(input_shape=(48, 48, 1)):
    return Sequential(
        [
            Input(shape=input_shape),
            Conv2D(6, (5, 5), padding="same", activation="relu"),
            MaxPool2D(pool_size=(2, 2)),
            Dropout(0.25),
            Conv2D(16, (5, 5), padding="same", activation="relu"),
            MaxPool2D(pool_size=(2, 2)),
            Dropout(0.25),
            Conv2D(64, (3, 3), activation="relu"),
            MaxPool2D(pool_size=(2, 2)),
            Dropout(0.25),
            Flatten(),
            Dense(128, activation="relu"),
            Dropout(0.5),
            Dense(2, activation="softmax"),
        ]
    )


model = create_model()
model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
model.summary()

In [ ]:
callbacks = [
    EarlyStopping(monitor="val_accuracy", min_delta=1e-4, patience=10, mode="max", restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_accuracy", factor=0.1, patience=5, mode="max", min_delta=1e-5, verbose=1),
]
history = model.fit(
    X_train, y_train, batch_size=8, epochs=50, validation_data=(X_val, y_val), callbacks=callbacks, verbose=2
)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8))
for ax, metric in zip(axes, ["loss", "accuracy"]):
    ax.plot(history.history[metric], label="train")
    ax.plot(history.history[f"val_{metric}"], label="validation")
    ax.set_title(metric)
    ax.legend()
    ax.grid(True)
plt.tight_layout()
plt.show()

## Evaluation on the test set

In [ ]:
p_test = model.predict(X_test, verbose=0)
y_true = y_test.argmax(axis=1)
y_pred = p_test.argmax(axis=1)

print(f"Test accuracy: {accuracy_score(y_true, y_pred):.3f}")
print(classification_report(y_true, y_pred, target_names=CLASSES))

sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt="d", xticklabels=CLASSES, yticklabels=CLASSES, cmap="Blues")
plt.title("Gabor CNN - test set")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## Save the model and test predictions

In [ ]:
model.save(MODELS / "image_model.keras")

image_predictions = pd.DataFrame(
    {
        "path": faces.loc[idx_test, "path"].astype(str).to_numpy(),
        "emotion": faces.loc[idx_test, "emotion"].to_numpy(),
        "anxiety_true": y_true,
        "anxiety_pred": y_pred,
        "p_anxiety": p_test[:, 1],
    }
)
image_predictions.to_csv(OUTPUTS / "image_predictions.csv", index=False)
image_predictions.head()

## Predict a single image

A new image must go through exactly the same preprocessing as the training data: grayscale, 48x48, Gabor filtering and scaling to [0, 1].

In [ ]:
def predict_image(path, model):
    face = load_face(path)
    p_anxiety = model.predict(to_model_input([face]), verbose=0)[0, 1]
    return face, p_anxiety


image_model = load_model(MODELS / "image_model.keras")
path = image_predictions.loc[0, "path"]
face, p_anxiety = predict_image(path, image_model)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(face, cmap="gray")
axes[0].set_title(f"{Path(path).name} ({image_predictions.loc[0, 'emotion']})")
axes[0].axis("off")
axes[1].bar(CLASSES, [1 - p_anxiety, p_anxiety])
axes[1].set_ylim(0, 1)
axes[1].set_ylabel("Probability")
plt.tight_layout()
plt.show()